In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
%%writefile cuda_program.cu
#include <iostream>
#include <chrono>
#include <cstdlib>
#include <ctime>

using namespace std;
using namespace chrono;


__global__ void vecAdd(int *a, int *b, int *c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n)
        c[i] = a[i] + b[i];
}


__global__ void matMul(int *A, int *B, int *C,int r1, int c1, int c2)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y; 
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < r1 && col < c2)
    {
        int sum = 0;

        for (int k = 0; k < c1; k++)
        {
            sum += A[row * c1 + k] *B[k * c2 + col];
        }

        C[row * c2 + col] = sum;
    }
}


void printVector(int *arr, int n)
{
    for (int i = 0; i < n; i++)
        cout << arr[i] << " ";

    cout << endl;
}


void printMatrix(int *arr, int r, int c)
{
    for (int i = 0; i < r; i++)
    {
        for (int j = 0; j < c; j++)
            cout << arr[i * c + j] << " ";

        cout << endl;
    }
}


int main()
{
    srand(time(0));

    cout << "\n===== VECTOR ADDITION =====\n";

    int n;

    cout << "Enter vector size: ";
    cin >> n;

    int *a = new int[n];
    int *b = new int[n];
    int *c = new int[n];

    if (n < 50)
    {
        cout << "Enter Vector A:\n";

        for (int i = 0; i < n; i++)
            cin >> a[i];

        cout << "Enter Vector B:\n";

        for (int i = 0; i < n; i++)
            cin >> b[i];
    }
    else
    {
        for (int i = 0; i < n; i++)
        {
            a[i] = rand() % 1000;
            b[i] = rand() % 1000;
        }

        cout << "Large random vectors generated.\n";
    }



    auto start = high_resolution_clock::now();

    for (int i = 0; i < n; i++)
        c[i] = a[i] + b[i];

    auto end = high_resolution_clock::now();

    if (n < 50)
    {
        cout << "\nCPU Output:\n";
        printVector(c, n);
    }

    cout << "CPU Vector Add Time: "<< duration_cast<microseconds>(end - start).count()<< " us\n";

  

    int *d_a, *d_b, *d_c;

    cudaMalloc(&d_a, n * sizeof(int));
    cudaMalloc(&d_b, n * sizeof(int));
    cudaMalloc(&d_c, n * sizeof(int));

    cudaMemcpy(d_a, a, n * sizeof(int),cudaMemcpyHostToDevice);

    cudaMemcpy(d_b, b, n * sizeof(int),cudaMemcpyHostToDevice);

    cudaEvent_t startEvent, stopEvent;

    cudaEventCreate(&startEvent);
    cudaEventCreate(&stopEvent);

    cudaEventRecord(startEvent);

    vecAdd<<<(n + 255) / 256, 256>>>(d_a, d_b, d_c, n);

    cudaEventRecord(stopEvent);
    cudaEventSynchronize(stopEvent);

    float gpuTime;

    cudaEventElapsedTime(&gpuTime,startEvent,stopEvent);

    cudaMemcpy(c, d_c,n * sizeof(int),cudaMemcpyDeviceToHost);

    if (n < 50)
    {
        cout << "\nGPU Output:\n";
        printVector(c, n);
    }

    cout << "GPU Vector Add Time: "
         << gpuTime << " ms\n";



    cout << "\n===== MATRIX MULTIPLICATION =====\n";

    int r1, c1, r2, c2;

    cout << "Enter rows and cols of Matrix A: ";
    cin >> r1 >> c1;

    cout << "Enter rows and cols of Matrix B: ";
    cin >> r2 >> c2;

    // Edge Case Validation
    if (c1 != r2)
    {
        cout << "\nInvalid Matrix Multiplication!\n";
        cout << "Columns of A must equal rows of B.\n";

        return 0;
    }

    int *A = new int[r1 * c1];
    int *B = new int[r2 * c2];
    int *C = new int[r1 * c2];


    if (r1 < 10 && c1 < 10 && r2 < 10 && c2 < 10)
    {
        cout << "\nEnter Matrix A:\n";

        for (int i = 0; i < r1 * c1; i++)
            cin >> A[i];

        cout << "\nEnter Matrix B:\n";

        for (int i = 0; i < r2 * c2; i++)
            cin >> B[i];
    }
    else
    {
        for (int i = 0; i < r1 * c1; i++)
            A[i] = rand() % 10;

        for (int i = 0; i < r2 * c2; i++)
            B[i] = rand() % 10;

        cout << "Large random matrices generated.\n";
    }



    start = high_resolution_clock::now();

    for (int i = 0; i < r1; i++)
    {
        for (int j = 0; j < c2; j++)
        {
            int sum = 0;

            for (int k = 0; k < c1; k++)
            {
                sum += A[i * c1 + k] *B[k * c2 + j];
            }

            C[i * c2 + j] = sum;
        }
    }

    end = high_resolution_clock::now();

    if (r1 < 10 && c1 < 10 && r2 < 10 && c2 < 10)
    {
        cout << "\nCPU Matrix Output:\n";
        printMatrix(C, r1, c2);
    }

    cout << "CPU Matrix Mul Time: "<< duration_cast<milliseconds>(end - start).count()<< " ms\n";



    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, r1 * c1 * sizeof(int));
    cudaMalloc(&d_B, r2 * c2 * sizeof(int));
    cudaMalloc(&d_C, r1 * c2 * sizeof(int));

    cudaMemcpy(d_A, A,r1 * c1 * sizeof(int),cudaMemcpyHostToDevice);

    cudaMemcpy(d_B, B,r2 * c2 * sizeof(int),cudaMemcpyHostToDevice);

    dim3 threads(16, 16);

    dim3 blocks((c2 + 15) / 16,(r1 + 15) / 16);

    cudaEventRecord(startEvent);

    matMul<<<blocks, threads>>>(d_A, d_B, d_C,r1, c1, c2);

    cudaEventRecord(stopEvent);
    cudaEventSynchronize(stopEvent);

    cudaEventElapsedTime(&gpuTime,startEvent,stopEvent);

    cudaMemcpy(C, d_C,r1 * c2 * sizeof(int),cudaMemcpyDeviceToHost);

    if (r1 < 10 && c1 < 10 &&r2 < 10 && c2 < 10)
    {
        cout << "\nGPU Matrix Output:\n";
        printMatrix(C, r1, c2);
    }

    cout << "GPU Matrix Mul Time: "
         << gpuTime << " ms\n";


    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    delete[] a;
    delete[] b;
    delete[] c;

    delete[] A;
    delete[] B;
    delete[] C;

    return 0;
}

Writing cuda_program.cu


In [3]:
!nvcc cuda_program.cu -o run
!./run

/bin/bash: line 1: nvcc: command not found
/bin/bash: line 1: ./run: No such file or directory
